# 01 — Wrangle & Feature Engineer

**Purpose:** Load validated transactions, add time/classification features,
build monthly aggregates per category.

**Inputs:** `data/interim/transactions_validated.parquet`  
**Outputs:**
- `data/interim/transactions_featured.parquet`
- `data/interim/monthly_aggregates.parquet`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.utils.logging_config import setup_notebook_logging
from src.analysis.feature_engineering import engineer_features, build_monthly_aggregates
from src.storage.local_writer import LocalWriter

logger = setup_notebook_logging()
writer = LocalWriter(project_root="..")

In [ ]:
# ── Load validated transactions ───────────────────────────────────────────────
df = writer.load_interim("transactions_validated")
print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
# ── Engineer features ─────────────────────────────────────────────────────────
df_featured = engineer_features(df)
print(f"\nNew columns added: {[c for c in df_featured.columns if c not in df.columns]}")
display(df_featured.head())

In [ ]:
# ── Transfers summary ─────────────────────────────────────────────────────────
n_transfers = df_featured["is_transfer"].sum()
print(f"Transfers (excluded from category analysis): {n_transfers} ({n_transfers/len(df_featured):.1%})")

In [ ]:
# ── Monthly aggregates ────────────────────────────────────────────────────────
monthly = build_monthly_aggregates(df_featured)
print(f"\nMonthly aggregates: {monthly.shape[0]} rows")
display(monthly.head(20))

In [ ]:
# ── Top categories by total spend (absolute value) ────────────────────────────
top_cats = (
    monthly.groupby("category")["total_amount"]
    .sum()
    .abs()
    .sort_values(ascending=False)
    .head(15)
)
display(top_cats.rename("absolute_spend").to_frame())

In [ ]:
# ── Net cash flow per month (income - expenses, excluding transfers) ───────────
net_monthly = (
    monthly.groupby("year_month")["total_amount"]
    .sum()
    .reset_index()
    .rename(columns={"total_amount": "net_cashflow"})
)

fig, ax = plt.subplots(figsize=(12, 4))
colors = net_monthly["net_cashflow"].map(lambda x: "steelblue" if x >= 0 else "tomato")
ax.bar(net_monthly["year_month"], net_monthly["net_cashflow"], color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Net Cash Flow per Month (excl. transfers)")
ax.set_xlabel("Month")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────────
writer.save_interim(df_featured, "transactions_featured")
writer.save_interim(monthly, "monthly_aggregates")
print("Saved:")
print("  data/interim/transactions_featured.parquet")
print("  data/interim/monthly_aggregates.parquet")